# Hallucination Detection Pipeline
This notebook tests whether LLM hallucinations are geometrically separable from correct answers
in the model's internal activation space.

1. **Setup** — Install dependencies, upload dataset, login to HuggingFace
2. **Extract** — Run LLM forward passes, record hidden states AND the model's predicted answer
3. **Dataset Analysis** — Hallucination rate, class balance, confidence distributions
4. **LR Probe** — Logistic regression layer sweep (hallucination vs correct labels)
5. **MLP Probe** — Non-linear probe with PCA preprocessing
6. **PCA + LR** — Dimensionality reduction then logistic regression
7. **Geometric Eval** — PCA visualization, distance analysis, and clustering
8. **Confidence Ablation** — Probe high-confidence vs low-confidence subsets
9. **Summary** — Compare all methods and download results

**Before running:** Set runtime to GPU via `Runtime > Change runtime type > T4 GPU`

## Step 0: Setup

In [ ]:
!pip install -q torch transformers accelerate pandas pyarrow scikit-learn matplotlib huggingface_hub bitsandbytes

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from google.colab import files
print("Upload your dataset.parquet file:")
uploaded = files.upload()
DATASET_PATH = list(uploaded.keys())[0]
print(f"Uploaded: {DATASET_PATH}")

In [ ]:
# ---- CONFIG ----
MODEL_NAME = "meta-llama/Llama-2-13b-hf"
FEATURES_DIR = "features"
BATCH_SIZE = 4
MAX_LENGTH = 256
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PROMPT_TEMPLATE = "True or False: {statement}\nAnswer:"
USE_4BIT = True
SEED = 42
N_FOLDS = 5
PROBE_C = 1.0
PCA_N_COMPONENTS = 256
MLP_PCA_COMPONENTS = 128
MLP_HIDDEN_SIZE = 32
MLP_DROPOUT = 0.3
MLP_LR = 1e-3
MLP_WEIGHT_DECAY = 1e-3
MLP_EPOCHS = 50
MLP_BATCH = 256
N_PAIRS = 10000
COV_REG = 1e-4

print(f"Model: {MODEL_NAME}")
print(f"Device: {DEVICE}")
print(f"4-bit quantization: {USE_4BIT}")
print(f"Prompt template: {PROMPT_TEMPLATE}")

## Step 1: Extract Features + Model Predictions
For each example, run a forward pass and capture:
- Hidden state at the last token for every layer
- The model's predicted answer (True/False) from next-token logits
- Whether the prediction matches ground truth (hallucinated or not)

In [ ]:
from __future__ import annotations
import json
from pathlib import Path
import numpy as np
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


def _infer_input_device(model, fallback_device: str) -> torch.device:
    if hasattr(model, "hf_device_map") and isinstance(model.hf_device_map, dict):
        for key in ["model.embed_tokens", "embed_tokens", "lm_head"]:
            if key in model.hf_device_map:
                dev = model.hf_device_map[key]
                if isinstance(dev, int):
                    return torch.device(f"cuda:{dev}")
                return torch.device(str(dev))
        first_dev = next(iter(model.hf_device_map.values()), None)
        if first_dev is not None:
            if isinstance(first_dev, int):
                return torch.device(f"cuda:{first_dev}")
            return torch.device(str(first_dev))
    return torch.device(fallback_device)


def _get_answer_token_ids(tokenizer):
    """Find token IDs for True/False/Yes/No in the tokenizer's vocabulary."""
    candidates = {
        "true": ["True", "true", "TRUE", " True", " true"],
        "false": ["False", "false", "FALSE", " False", " false"],
    }
    token_ids = {}
    for label, variants in candidates.items():
        ids = set()
        for v in variants:
            encoded = tokenizer.encode(v, add_special_tokens=False)
            if len(encoded) >= 1:
                ids.add(encoded[0])
        token_ids[label] = list(ids)
    print(f"  Answer token IDs — true: {token_ids['true']}, false: {token_ids['false']}")
    return token_ids


def extract_features_with_predictions(
    model_name, dataset_path, out_dir, batch_size, max_length,
    device, prompt_template, use_4bit=False,
):
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    df = pd.read_parquet(dataset_path)
    required = {"id", "domain", "label", "statement"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {', '.join(sorted(missing))}")

    texts = [prompt_template.format(statement=str(s)) for s in df["statement"].tolist()]
    ground_truth = df["label"].astype(int).to_numpy()
    domains = df["domain"].astype(str).to_numpy()
    ids = df["id"].astype(str).to_numpy()
    print(f"Loaded {len(texts)} examples")
    print(f"Example prompt: {texts[0]!r}")

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    answer_ids = _get_answer_token_ids(tokenizer)

    dtype = torch.float16 if device.startswith("cuda") else torch.float32

    if use_4bit and device.startswith("cuda"):
        print("Loading model with 4-bit quantization (NF4)...")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_name, quantization_config=bnb_config,
            output_hidden_states=True, device_map="auto",
        )
    elif device.startswith("cuda"):
        print(f"Loading model in {dtype}...")
        model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype=dtype,
            output_hidden_states=True, device_map="auto",
        )
    else:
        print(f"Loading model in {dtype}...")
        model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype=dtype, output_hidden_states=True,
        )
        model.to(device)
    model.eval()
    input_device = _infer_input_device(model, fallback_device=device)
    print(f"Model loaded. Input device: {input_device}")

    sample = tokenizer("test", return_tensors="pt")
    sample = {k: v.to(input_device) for k, v in sample.items()}
    with torch.no_grad():
        out = model(**sample)
    num_layers = len(out.hidden_states)
    hidden_dim = out.hidden_states[0].shape[-1]
    print(f"Detected {num_layers} layers, hidden_dim={hidden_dim}")

    feats_by_layer = [[] for _ in range(num_layers)]
    model_preds = []      # 1=true, 0=false
    model_confs = []      # P(chosen answer)
    pred_raw_probs = []   # (p_true, p_false) for each example
    total_batches = (len(texts) + batch_size - 1) // batch_size

    for batch_num, i in enumerate(range(0, len(texts), batch_size)):
        if batch_num % 20 == 0:
            print(f"  Batch {batch_num + 1}/{total_batches}")
        batch_texts = texts[i : i + batch_size]
        enc = tokenizer(
            batch_texts, return_tensors="pt", padding=True,
            truncation=True, max_length=max_length,
        )
        enc = {k: v.to(input_device) for k, v in enc.items()}

        with torch.no_grad():
            out = model(**enc)
            hs = out.hidden_states
            logits = out.logits  # (batch, seq_len, vocab_size)

        attn = enc["attention_mask"]
        last_idx = (attn.sum(dim=1) - 1).clamp(min=0)
        batch_idx = torch.arange(last_idx.shape[0], device=last_idx.device)

        for layer_idx in range(num_layers):
            h_last = hs[layer_idx][batch_idx, last_idx, :]
            feats_by_layer[layer_idx].append(h_last.detach().float().cpu().numpy())

        next_logits = logits[batch_idx, last_idx, :]  # (batch, vocab)
        probs = torch.softmax(next_logits.float(), dim=-1)

        for b in range(len(batch_texts)):
            p_true = sum(float(probs[b, tid]) for tid in answer_ids["true"])
            p_false = sum(float(probs[b, tid]) for tid in answer_ids["false"])
            pred_raw_probs.append((p_true, p_false))

            if p_true >= p_false:
                model_preds.append(1)
                model_confs.append(p_true / (p_true + p_false + 1e-12))
            else:
                model_preds.append(0)
                model_confs.append(p_false / (p_true + p_false + 1e-12))

    model_preds = np.array(model_preds, dtype=int)
    model_confs = np.array(model_confs, dtype=np.float32)
    hallucinated = (model_preds != ground_truth).astype(int)

    n_correct = int((hallucinated == 0).sum())
    n_halluc = int((hallucinated == 1).sum())
    print(f"\n--- Prediction Summary ---")
    print(f"Model accuracy on dataset: {n_correct / len(hallucinated):.4f}")
    print(f"Correct: {n_correct}  |  Hallucinated: {n_halluc}")
    print(f"Hallucination rate: {n_halluc / len(hallucinated):.4f}")

    print("\nSaving features...")
    meta = {
        "model_name": model_name, "num_layers": num_layers, "hidden_dim": hidden_dim,
        "n_examples": len(texts), "max_length": max_length, "batch_size": batch_size,
        "device": device, "prompt_template": prompt_template,
        "n_correct": n_correct, "n_hallucinated": n_halluc,
        "model_accuracy": float(n_correct / len(hallucinated)),
    }
    with (out_path / "meta.json").open("w") as f:
        json.dump(meta, f, indent=2)

    np.savez_compressed(
        out_path / "labels_and_preds.npz",
        ground_truth=ground_truth,
        model_preds=model_preds,
        model_confs=model_confs,
        hallucinated=hallucinated,
        domain=domains,
        id=ids,
    )

    for layer_idx in range(num_layers):
        x = np.concatenate(feats_by_layer[layer_idx], axis=0)
        np.savez_compressed(out_path / f"layer_{layer_idx:02d}.npz", X=x)
        print(f"  Saved layer_{layer_idx:02d}.npz  shape={x.shape}")

    print(f"\nExtraction complete. {num_layers} layers saved to {out_dir}/")
    return num_layers, hidden_dim

In [ ]:
num_layers, hidden_dim = extract_features_with_predictions(
    model_name=MODEL_NAME,
    dataset_path=DATASET_PATH,
    out_dir=FEATURES_DIR,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
    device=DEVICE,
    prompt_template=PROMPT_TEMPLATE,
    use_4bit=USE_4BIT,
)

## Step 2: Dataset Analysis
Examine the hallucination dataset before probing.

In [ ]:
import matplotlib.pyplot as plt

shared = np.load(Path(FEATURES_DIR) / "labels_and_preds.npz", allow_pickle=True)
ground_truth = shared["ground_truth"]
model_preds = shared["model_preds"]
model_confs = shared["model_confs"]
hallucinated = shared["hallucinated"]

print("=" * 50)
print("DATASET ANALYSIS")
print("=" * 50)
print(f"Total examples: {len(hallucinated)}")
print(f"Correct:        {(hallucinated == 0).sum()} ({(hallucinated == 0).mean():.1%})")
print(f"Hallucinated:   {(hallucinated == 1).sum()} ({(hallucinated == 1).mean():.1%})")
print()
print("Breakdown by ground truth label:")
for gt_label in [0, 1]:
    mask = ground_truth == gt_label
    n = mask.sum()
    n_correct = ((model_preds == ground_truth) & mask).sum()
    label_name = "True" if gt_label == 1 else "False"
    print(f"  GT={label_name}: {n} total, {n_correct} correct ({n_correct/n:.1%}), "
          f"{n - n_correct} hallucinated ({(n - n_correct)/n:.1%})")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(model_confs[hallucinated == 0], bins=50, alpha=0.7, label="Correct", density=True)
axes[0].hist(model_confs[hallucinated == 1], bins=50, alpha=0.7, label="Hallucinated", density=True)
axes[0].set_xlabel("Model Confidence")
axes[0].set_ylabel("Density")
axes[0].set_title("Confidence Distribution")
axes[0].legend()

axes[1].bar(["Correct", "Hallucinated"], [(hallucinated == 0).sum(), (hallucinated == 1).sum()])
axes[1].set_ylabel("Count")
axes[1].set_title("Class Balance")

fig.tight_layout()
plt.show()

print(f"\nMean confidence — Correct: {model_confs[hallucinated == 0].mean():.4f}, "
      f"Hallucinated: {model_confs[hallucinated == 1].mean():.4f}")

## Step 3: Layer Sweep — Logistic Regression
Train a logistic regression probe on each layer using hallucination labels.
Uses stratified 5-fold CV and class-balanced weights.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold


def eval_probe(clf, X, y):
    probs = clf.predict_proba(X)[:, 1]
    preds = (probs >= 0.5).astype(int)
    metrics = {
        "acc": float(accuracy_score(y, preds)),
        "bal_acc": float(balanced_accuracy_score(y, preds)),
    }
    try:
        metrics["auroc"] = float(roc_auc_score(y, probs))
    except ValueError:
        metrics["auroc"] = float("nan")
    return metrics


def run_lr_sweep(features_dir, y, n_folds, seed, C):
    features_path = Path(features_dir)
    layer_files = sorted(features_path.glob("layer_*.npz"), key=lambda p: int(p.stem.split("_")[1]))
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    print(f"LR Probe: {len(layer_files)} layers, {n_folds}-fold CV, C={C}, class_weight=balanced")

    rows = []
    for layer_file in layer_files:
        layer_idx = int(layer_file.stem.split("_")[1])
        X = np.load(layer_file)["X"]

        fold_metrics = []
        for train_idx, test_idx in skf.split(X, y):
            clf = LogisticRegression(
                penalty="l2", C=C, solver="lbfgs", max_iter=200,
                class_weight="balanced", random_state=seed,
            )
            clf.fit(X[train_idx], y[train_idx])
            train_m = eval_probe(clf, X[train_idx], y[train_idx])
            test_m = eval_probe(clf, X[test_idx], y[test_idx])
            fold_metrics.append({"train": train_m, "test": test_m})

        row = {
            "layer": layer_idx,
            "train_acc": np.mean([f["train"]["acc"] for f in fold_metrics]),
            "train_auroc": np.mean([f["train"]["auroc"] for f in fold_metrics]),
            "test_acc": np.mean([f["test"]["acc"] for f in fold_metrics]),
            "test_bal_acc": np.mean([f["test"]["bal_acc"] for f in fold_metrics]),
            "test_auroc": np.mean([f["test"]["auroc"] for f in fold_metrics]),
            "test_auroc_std": np.std([f["test"]["auroc"] for f in fold_metrics]),
        }
        rows.append(row)
        print(f"  layer={layer_idx:02d}  train_auroc={row['train_auroc']:.4f}  "
              f"test_auroc={row['test_auroc']:.4f} ± {row['test_auroc_std']:.4f}  "
              f"test_bal_acc={row['test_bal_acc']:.4f}")

    return pd.DataFrame(rows).sort_values("layer").reset_index(drop=True)

In [ ]:
lr_df = run_lr_sweep(FEATURES_DIR, hallucinated, n_folds=N_FOLDS, seed=SEED, C=PROBE_C)

lr_best = lr_df.loc[lr_df["test_auroc"].idxmax()]
BEST_LAYER = int(lr_best["layer"])
print(f"\nBest layer: {BEST_LAYER} (test_auroc={lr_best['test_auroc']:.4f})")

lr_df.to_csv("lr_halluc_sweep.csv", index=False)
print("Saved lr_halluc_sweep.csv")
lr_df

## Step 4: MLP Probe (Non-linear)
PCA to 128 dims, then a small MLP (hidden=32) with dropout and balanced sampling.
Tests whether hallucinations are separable non-linearly.

In [ ]:
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
from sklearn.decomposition import PCA


class _MLPNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


class MLPProbe:
    def __init__(self, input_dim, hidden_dim=32, dropout=0.3, lr=1e-3,
                 weight_decay=1e-3, epochs=50, batch_size=256, seed=42, device="cpu"):
        torch.manual_seed(seed)
        self.device = torch.device(device)
        self.model = _MLPNet(input_dim, hidden_dim, dropout).to(self.device)
        self.lr = lr
        self.weight_decay = weight_decay
        self.epochs = epochs
        self.batch_size = batch_size
        self.seed = seed

    def fit(self, X, y):
        self.model.train()
        ds = TensorDataset(torch.tensor(X, dtype=torch.float32),
                           torch.tensor(y, dtype=torch.float32))

        n_pos = float(y.sum())
        n_neg = float(len(y) - n_pos)
        sample_weights = np.where(y == 1, 1.0 / n_pos, 1.0 / n_neg)
        sampler = WeightedRandomSampler(sample_weights, num_samples=len(y), replacement=True)
        loader = DataLoader(ds, batch_size=self.batch_size, sampler=sampler)

        opt = torch.optim.Adam(self.model.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        loss_fn = nn.BCEWithLogitsLoss()

        best_loss = float("inf")
        patience_counter = 0
        best_state = None

        for epoch in range(self.epochs):
            epoch_loss = 0.0
            n_batches = 0
            for xb, yb in loader:
                xb, yb = xb.to(self.device), yb.to(self.device)
                opt.zero_grad()
                loss = loss_fn(self.model(xb), yb)
                loss.backward()
                opt.step()
                epoch_loss += loss.item()
                n_batches += 1
            avg_loss = epoch_loss / max(n_batches, 1)
            if avg_loss < best_loss - 1e-4:
                best_loss = avg_loss
                patience_counter = 0
                best_state = {k: v.clone() for k, v in self.model.state_dict().items()}
            else:
                patience_counter += 1
            if patience_counter >= 5:
                break

        if best_state is not None:
            self.model.load_state_dict(best_state)
        return self

    def predict_proba(self, X):
        self.model.eval()
        with torch.no_grad():
            logits = self.model(torch.tensor(X, dtype=torch.float32).to(self.device))
            p1 = torch.sigmoid(logits).cpu().numpy()
        return np.column_stack([1 - p1, p1])


def run_mlp_sweep(features_dir, y, n_folds, seed, pca_components,
                  hidden_size, dropout, device):
    features_path = Path(features_dir)
    layer_files = sorted(features_path.glob("layer_*.npz"),
                         key=lambda p: int(p.stem.split("_")[1]))
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    print(f"MLP Probe: {len(layer_files)} layers, PCA({pca_components})->MLP({hidden_size}), "
          f"dropout={dropout}, {n_folds}-fold CV")

    rows = []
    for layer_file in layer_files:
        layer_idx = int(layer_file.stem.split("_")[1])
        X_full = np.load(layer_file)["X"]

        fold_metrics = []
        for train_idx, test_idx in skf.split(X_full, y):
            pca = PCA(n_components=min(pca_components, X_full.shape[1]), random_state=seed)
            pca.fit(X_full[train_idx])
            X_train = pca.transform(X_full[train_idx])
            X_test = pca.transform(X_full[test_idx])

            clf = MLPProbe(
                input_dim=X_train.shape[1], hidden_dim=hidden_size, dropout=dropout,
                lr=MLP_LR, weight_decay=MLP_WEIGHT_DECAY, epochs=MLP_EPOCHS,
                batch_size=MLP_BATCH, seed=seed, device=device,
            ).fit(X_train, y[train_idx])

            train_m = eval_probe(clf, X_train, y[train_idx])
            test_m = eval_probe(clf, X_test, y[test_idx])
            fold_metrics.append({"train": train_m, "test": test_m})

        row = {
            "layer": layer_idx,
            "train_acc": np.mean([f["train"]["acc"] for f in fold_metrics]),
            "train_auroc": np.mean([f["train"]["auroc"] for f in fold_metrics]),
            "test_acc": np.mean([f["test"]["acc"] for f in fold_metrics]),
            "test_bal_acc": np.mean([f["test"]["bal_acc"] for f in fold_metrics]),
            "test_auroc": np.mean([f["test"]["auroc"] for f in fold_metrics]),
            "test_auroc_std": np.std([f["test"]["auroc"] for f in fold_metrics]),
        }
        rows.append(row)
        print(f"  layer={layer_idx:02d}  train_auroc={row['train_auroc']:.4f}  "
              f"test_auroc={row['test_auroc']:.4f} ± {row['test_auroc_std']:.4f}  "
              f"test_bal_acc={row['test_bal_acc']:.4f}")

    return pd.DataFrame(rows).sort_values("layer").reset_index(drop=True)

In [ ]:
mlp_df = run_mlp_sweep(
    FEATURES_DIR, hallucinated, n_folds=N_FOLDS, seed=SEED,
    pca_components=MLP_PCA_COMPONENTS, hidden_size=MLP_HIDDEN_SIZE,
    dropout=MLP_DROPOUT, device=DEVICE,
)

mlp_best = mlp_df.loc[mlp_df["test_auroc"].idxmax()]
print(f"\nMLP best layer: {int(mlp_best['layer'])} (test_auroc={mlp_best['test_auroc']:.4f})")
print(f"LR  best layer: {BEST_LAYER} (test_auroc={lr_best['test_auroc']:.4f})")
print(f"MLP improvement over LR: {mlp_best['test_auroc'] - lr_best['test_auroc']:+.4f}")

mlp_df.to_csv("mlp_halluc_sweep.csv", index=False)
print("Saved mlp_halluc_sweep.csv")
mlp_df

## Step 5: PCA + Logistic Regression
Reduce to 256 principal components, then logistic regression.

In [ ]:
def run_pca_lr_sweep(features_dir, y, n_folds, seed, C, n_components):
    features_path = Path(features_dir)
    layer_files = sorted(features_path.glob("layer_*.npz"),
                         key=lambda p: int(p.stem.split("_")[1]))
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    print(f"PCA+LR Probe: {len(layer_files)} layers, PCA({n_components}), C={C}, {n_folds}-fold CV")

    rows = []
    for layer_file in layer_files:
        layer_idx = int(layer_file.stem.split("_")[1])
        X_full = np.load(layer_file)["X"]

        fold_metrics = []
        for train_idx, test_idx in skf.split(X_full, y):
            pca = PCA(n_components=min(n_components, X_full.shape[1]), random_state=seed)
            pca.fit(X_full[train_idx])
            X_train = pca.transform(X_full[train_idx])
            X_test = pca.transform(X_full[test_idx])

            clf = LogisticRegression(
                penalty="l2", C=C, solver="lbfgs", max_iter=200,
                class_weight="balanced", random_state=seed,
            )
            clf.fit(X_train, y[train_idx])
            train_m = eval_probe(clf, X_train, y[train_idx])
            test_m = eval_probe(clf, X_test, y[test_idx])
            fold_metrics.append({"train": train_m, "test": test_m})

        row = {
            "layer": layer_idx,
            "train_acc": np.mean([f["train"]["acc"] for f in fold_metrics]),
            "train_auroc": np.mean([f["train"]["auroc"] for f in fold_metrics]),
            "test_acc": np.mean([f["test"]["acc"] for f in fold_metrics]),
            "test_bal_acc": np.mean([f["test"]["bal_acc"] for f in fold_metrics]),
            "test_auroc": np.mean([f["test"]["auroc"] for f in fold_metrics]),
            "test_auroc_std": np.std([f["test"]["auroc"] for f in fold_metrics]),
        }
        rows.append(row)
        print(f"  layer={layer_idx:02d}  train_auroc={row['train_auroc']:.4f}  "
              f"test_auroc={row['test_auroc']:.4f} ± {row['test_auroc_std']:.4f}  "
              f"test_bal_acc={row['test_bal_acc']:.4f}")

    return pd.DataFrame(rows).sort_values("layer").reset_index(drop=True)

In [ ]:
pca_lr_df = run_pca_lr_sweep(
    FEATURES_DIR, hallucinated, n_folds=N_FOLDS, seed=SEED,
    C=PROBE_C, n_components=PCA_N_COMPONENTS,
)

pca_best = pca_lr_df.loc[pca_lr_df["test_auroc"].idxmax()]
print(f"\nPCA+LR best layer: {int(pca_best['layer'])} (test_auroc={pca_best['test_auroc']:.4f})")
print(f"Full LR best layer: {BEST_LAYER} (test_auroc={lr_best['test_auroc']:.4f})")

pca_lr_df.to_csv("pca_lr_halluc_sweep.csv", index=False)
print("Saved pca_lr_halluc_sweep.csv")
pca_lr_df

## Step 6: Geometric Evaluation
PCA visualization, distance analysis, and clustering on the best layer.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_mutual_info_score


def sample_pairs(a_idx, b_idx, n_pairs, seed):
    rng = np.random.RandomState(seed)
    pairs, seen = [], set()
    if len(a_idx) == 0 or len(b_idx) == 0:
        return pairs
    for _ in range(n_pairs * 10 + 100):
        if len(pairs) >= n_pairs:
            break
        i = int(a_idx[rng.randint(0, len(a_idx))])
        j = int(b_idx[rng.randint(0, len(b_idx))])
        if i == j:
            continue
        key = (min(i, j), max(i, j))
        if key not in seen:
            seen.add(key)
            pairs.append(key)
    return pairs


def cosine_distance(x, y):
    denom = float(np.linalg.norm(x) * np.linalg.norm(y))
    if denom <= 1e-12:
        return float("nan")
    return float(1.0 - np.dot(x, y) / denom)


def mahalanobis_distance(x, y, inv_cov):
    d = x - y
    v = float(np.dot(np.dot(d, inv_cov), d))
    return float(np.sqrt(max(v, 0.0)))


def summary_stats(values):
    arr = np.array(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        return {"n": 0, "mean": float("nan"), "std": float("nan")}
    return {"n": len(arr), "mean": float(np.mean(arr)), "std": float(np.std(arr))}


def run_geometric_eval(features_dir, y, layer, n_pairs, seed, cov_reg):
    base = Path(features_dir)
    X = np.load(base / f"layer_{int(layer):02d}.npz")["X"]

    idx_correct = np.where(y == 0)[0]
    idx_halluc = np.where(y == 1)[0]

    # --- PCA Visualization ---
    pca = PCA(n_components=2, random_state=seed)
    proj = pca.fit_transform(X)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(proj[idx_correct, 0], proj[idx_correct, 1], s=8, alpha=0.4, label="Correct")
    ax.scatter(proj[idx_halluc, 0], proj[idx_halluc, 1], s=8, alpha=0.4, label="Hallucinated")
    ax.set_title(f"PCA Projection — Layer {int(layer):02d}")
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.2f}% var)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.2f}% var)")
    ax.legend(loc="best")
    ax.grid(alpha=0.2)
    fig.tight_layout()
    plt.show()

    # --- Distance Analysis ---
    pairs_cc = sample_pairs(idx_correct, idx_correct, n_pairs, seed + 1)
    pairs_hh = sample_pairs(idx_halluc, idx_halluc, n_pairs, seed + 2)
    pairs_ch = sample_pairs(idx_correct, idx_halluc, n_pairs, seed + 3)

    cos_cc = [cosine_distance(X[i], X[j]) for i, j in pairs_cc]
    cos_hh = [cosine_distance(X[i], X[j]) for i, j in pairs_hh]
    cos_ch = [cosine_distance(X[i], X[j]) for i, j in pairs_ch]

    print(f"\n--- Distance Summary (Layer {layer}) ---")
    print(f"Cosine | within-correct: {summary_stats(cos_cc)['mean']:.4f}  "
          f"within-halluc: {summary_stats(cos_hh)['mean']:.4f}  "
          f"between: {summary_stats(cos_ch)['mean']:.4f}")

    # --- Clustering ---
    print(f"\n--- Clustering Analysis (Layer {layer}) ---")

    pca_cluster = PCA(n_components=min(64, X.shape[1]), random_state=seed)
    X_reduced = pca_cluster.fit_transform(X)

    km = KMeans(n_clusters=2, random_state=seed, n_init=10)
    km_labels = km.fit_predict(X_reduced)
    km_ami = adjusted_mutual_info_score(y, km_labels)

    km_purity = 0
    for c in range(2):
        cluster_mask = km_labels == c
        if cluster_mask.sum() > 0:
            km_purity += max((y[cluster_mask] == 0).sum(), (y[cluster_mask] == 1).sum())
    km_purity /= len(y)

    gmm = GaussianMixture(n_components=2, random_state=seed, covariance_type="full")
    gmm_labels = gmm.fit_predict(X_reduced)
    gmm_ami = adjusted_mutual_info_score(y, gmm_labels)

    print(f"K-Means (k=2):  AMI={km_ami:.4f}, Purity={km_purity:.4f}")
    print(f"GMM (k=2):      AMI={gmm_ami:.4f}")
    print(f"(AMI=0 means clusters don't align with labels; AMI=1 means perfect alignment)")

    return {
        "layer": layer,
        "pca_var_ratio": pca.explained_variance_ratio_.tolist(),
        "cosine": {
            "within_correct": summary_stats(cos_cc),
            "within_halluc": summary_stats(cos_hh),
            "between": summary_stats(cos_ch),
        },
        "kmeans_ami": km_ami, "kmeans_purity": km_purity,
        "gmm_ami": gmm_ami,
    }

In [ ]:
print(f"Running geometric eval on best layer: {BEST_LAYER}")
geo_results = run_geometric_eval(
    FEATURES_DIR, hallucinated, layer=BEST_LAYER,
    n_pairs=N_PAIRS, seed=SEED, cov_reg=COV_REG,
)

## Step 7: Confidence-Conditioned Ablation
Split data by model confidence and probe each subset.
The most interesting case: high-confidence hallucinations (model is confident but wrong).

In [ ]:
def run_confidence_ablation(features_dir, hallucinated, model_confs, layer, n_folds, seed, C):
    base = Path(features_dir)
    X = np.load(base / f"layer_{int(layer):02d}.npz")["X"]
    y = hallucinated

    median_conf = np.median(model_confs)
    high_mask = model_confs >= median_conf
    low_mask = model_confs < median_conf

    results = {}
    for label, mask in [("all", np.ones(len(y), dtype=bool)),
                        ("high_confidence", high_mask),
                        ("low_confidence", low_mask)]:
        X_sub = X[mask]
        y_sub = y[mask]

        n_halluc = (y_sub == 1).sum()
        n_correct = (y_sub == 0).sum()

        if n_halluc < n_folds or n_correct < n_folds:
            print(f"  {label}: skipped (too few examples in one class: "
                  f"correct={n_correct}, halluc={n_halluc})")
            continue

        skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
        fold_aurocs = []
        fold_bal_accs = []
        for train_idx, test_idx in skf.split(X_sub, y_sub):
            clf = LogisticRegression(
                penalty="l2", C=C, solver="lbfgs", max_iter=200,
                class_weight="balanced", random_state=seed,
            )
            clf.fit(X_sub[train_idx], y_sub[train_idx])
            m = eval_probe(clf, X_sub[test_idx], y_sub[test_idx])
            fold_aurocs.append(m["auroc"])
            fold_bal_accs.append(m["bal_acc"])

        mean_auroc = np.mean(fold_aurocs)
        std_auroc = np.std(fold_aurocs)
        mean_bal_acc = np.mean(fold_bal_accs)
        results[label] = {"auroc": mean_auroc, "auroc_std": std_auroc,
                          "bal_acc": mean_bal_acc, "n": int(mask.sum()),
                          "n_halluc": int(n_halluc), "n_correct": int(n_correct)}
        print(f"  {label:20s}  n={mask.sum():5d} (halluc={n_halluc})  "
              f"auroc={mean_auroc:.4f} ± {std_auroc:.4f}  bal_acc={mean_bal_acc:.4f}")

    return results

In [ ]:
print(f"Confidence ablation on best layer: {BEST_LAYER}")
print(f"Confidence median: {np.median(model_confs):.4f}")
conf_results = run_confidence_ablation(
    FEATURES_DIR, hallucinated, model_confs,
    layer=BEST_LAYER, n_folds=N_FOLDS, seed=SEED, C=PROBE_C,
)

## Step 8: Summary & Download
Compare all probe methods and download results.

In [ ]:
print("=" * 75)
print("HALLUCINATION DETECTION — PROBE COMPARISON")
print("=" * 75)

comparison = pd.DataFrame({
    "Method": [
        "Logistic Regression",
        f"MLP (PCA {MLP_PCA_COMPONENTS}→hidden {MLP_HIDDEN_SIZE})",
        f"PCA({PCA_N_COMPONENTS})+LR",
    ],
    "Best Layer": [BEST_LAYER, int(mlp_best["layer"]), int(pca_best["layer"])],
    "Test AUROC": [lr_best["test_auroc"], mlp_best["test_auroc"], pca_best["test_auroc"]],
    "Test Bal Acc": [lr_best["test_bal_acc"], mlp_best["test_bal_acc"], pca_best["test_bal_acc"]],
    "Train AUROC": [lr_best["train_auroc"], mlp_best["train_auroc"], pca_best["train_auroc"]],
    "Overfit Gap": [
        lr_best["train_auroc"] - lr_best["test_auroc"],
        mlp_best["train_auroc"] - mlp_best["test_auroc"],
        pca_best["train_auroc"] - pca_best["test_auroc"],
    ],
})
print(comparison.to_string(index=False))

print(f"\n" + "=" * 75)
print("GEOMETRIC ANALYSIS")
print("=" * 75)
print(f"K-Means AMI:  {geo_results['kmeans_ami']:.4f}  (0 = no alignment, 1 = perfect)")
print(f"K-Means Purity: {geo_results['kmeans_purity']:.4f}")
print(f"GMM AMI:      {geo_results['gmm_ami']:.4f}")

print(f"\n" + "=" * 75)
print("INTERPRETATION")
print("=" * 75)
best_auroc = max(lr_best['test_auroc'], mlp_best['test_auroc'], pca_best['test_auroc'])
if best_auroc > 0.65:
    print(f"Best AUROC = {best_auroc:.4f}: Moderate-to-strong hallucination signal detected.")
elif best_auroc > 0.55:
    print(f"Best AUROC = {best_auroc:.4f}: Weak hallucination signal — slightly above chance.")
else:
    print(f"Best AUROC = {best_auroc:.4f}: No meaningful hallucination signal detected.")

mlp_gain = mlp_best['test_auroc'] - lr_best['test_auroc']
if mlp_gain > 0.03:
    print(f"MLP outperforms LR by {mlp_gain:+.4f}: hallucination signal may be non-linear.")
else:
    print(f"MLP vs LR gap = {mlp_gain:+.4f}: signal is roughly linear (if present).")

if geo_results['kmeans_ami'] > 0.05:
    print(f"Clustering shows alignment (AMI={geo_results['kmeans_ami']:.4f}): "
          f"hallucinations may occupy a distinct region.")
else:
    print(f"Clustering shows no alignment (AMI={geo_results['kmeans_ami']:.4f}): "
          f"hallucinations are not clustered separately.")

if conf_results.get("high_confidence"):
    hc = conf_results["high_confidence"]
    print(f"\nHigh-confidence subset: AUROC={hc['auroc']:.4f} "
          f"(n={hc['n']}, {hc['n_halluc']} hallucinated)")
    if hc['auroc'] > best_auroc + 0.02:
        print("  -> Signal is STRONGER in high-confidence examples: "
              "model may 'know' it's wrong even when confident.")
    elif hc['auroc'] < best_auroc - 0.02:
        print("  -> Signal is WEAKER in high-confidence examples: "
              "model genuinely doesn't know when it's confidently wrong.")

In [ ]:
import shutil

shutil.make_archive("features", "zip", ".", FEATURES_DIR)
print("Created features.zip")

from google.colab import files
for f in ["lr_halluc_sweep.csv", "mlp_halluc_sweep.csv",
          "pca_lr_halluc_sweep.csv", "features.zip"]:
    print(f"Downloading {f}...")
    files.download(f)